# NIDS Feature Selection GWO

Grey wolf optimization feature selection on a synthetic intrusion classification dataset. Edit data and search parameters; validation accuracy is balanced against selected subset size. This notebook runs without external datasets.

Run the code cell to generate **`nids-feature-selection-grey-wolf.png`** in the repository folder.

In [ ]:
# Editable inputs
SAMPLES,FEATURES,INFORMATIVE=240,18,5
POPULATION,ITERATIONS,SEED=28,100,21
MIN_FEATURES,MAX_FEATURES=1,10
OUTPUT="nids-feature-selection-grey-wolf.png"
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(SEED)
if not (1<=INFORMATIVE<=FEATURES): raise ValueError("INFORMATIVE must be between 1 and FEATURES")
y=np.repeat([0,1],SAMPLES//2); y=np.r_[y,np.zeros(SAMPLES-len(y),dtype=int)]; X=rng.normal(size=(SAMPLES,FEATURES)); X[:,:INFORMATIVE]+=(2*y-1)[:,None]*1.1; ix=rng.permutation(SAMPLES); tr,te=ix[:int(.75*SAMPLES)],ix[int(.75*SAMPLES):]
def fit(m):
 ids=np.flatnonzero(m)
 if not len(ids): return 0.
 means=np.array([X[tr[y[tr]==c]][:,ids].mean(0) for c in (0,1)]); pred=np.argmin(((X[te][:,ids][:,None,:]-means[None,:,:])**2).sum(2),axis=1); return float(np.mean(pred==y[te])-.02*len(ids)/FEATURES)
def repair(m):
 m=m.copy(); lo=max(1,MIN_FEATURES); hi=min(FEATURES,MAX_FEATURES)
 while m.sum()<lo:m[rng.integers(FEATURES)]=1
 if m.sum()>hi: m[rng.choice(np.flatnonzero(m),m.sum()-hi,False)]=0
 return m.astype(bool)
wolves=rng.random((POPULATION,FEATURES)); best=None; bestscore=-1; hist=[]
for t in range(ITERATIONS):
 masks=np.array([repair(w>=.5) for w in wolves]); scores=np.array([fit(m) for m in masks]); leaders=wolves[np.argsort(scores)[-3:]]
 if scores.max()>bestscore: best=masks[scores.argmax()].copy();bestscore=scores.max()
 a=2*(1-t/max(1,ITERATIONS-1)); updated=np.empty_like(wolves)
 for i,w in enumerate(wolves):
  proposals=[]
  for leader in leaders:
   A=2*a*rng.random(FEATURES)-a; C=2*rng.random(FEATURES); proposals.append(leader-A*np.abs(C*leader-w))
  updated[i]=np.clip(np.mean(proposals,axis=0),0,1)
 wolves=updated; hist.append(bestscore)
print("Selected feature indices:",np.flatnonzero(best).tolist()," adjusted score:",round(bestscore,3))
fig,(a,b)=plt.subplots(1,2,figsize=(11,4)); a.bar(range(FEATURES),best,color=["tomato" if i<INFORMATIVE else "steelblue" for i in range(FEATURES)]); a.set(title="Selected features (informative in red)",xlabel="Feature",ylabel="Selected"); b.plot(hist); b.set(title="Best adjusted score",xlabel="Iteration",ylabel="Score"); b.grid(alpha=.2); fig.tight_layout(); fig.savefig(OUTPUT,dpi=160); plt.show(); print("Saved",OUTPUT)